In [11]:
import torch
from check import setup_torch
# Dummy image-like data: batch=4, channels=1, height=256, width=256
data_dummy = torch.randn(4, 1, 256, 256, dtype=torch.float32)

device = setup_torch()
data_dummy.shape

✅ Using CUDA globally
GPU: NVIDIA GeForce RTX 4090


torch.Size([4, 1, 256, 256])

In [10]:
data_tensor = torch.load("../../json_data/dataset.pt", map_location=device)
data_tensor = [t.flatten() for t in data_tensor]
data_tensor = [t for t in data_tensor if t.numel() == 768000]
data_tensor = [t.reshape((128,6000)) for t in data_tensor]
data_tensor = torch.stack(data_tensor)
data_tensor.shape

torch.Size([103, 128, 6000])

In [27]:
from models.Data_Load import DataLoader_AutoEncoder, Dataset_

dataset = Dataset_(data_tensor)

train_size = 80
test_size = len(dataset) - train_size
gen = torch.Generator(device=device)
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size], generator=gen
)

dataLoader = DataLoader_AutoEncoder(train_dataset, batch_size=10, shuffle=True)
dataLoader_test = DataLoader_AutoEncoder(test_dataset, batch_size=10, shuffle=False)

In [28]:
from models.Autencoder import AutoEncoder

latent_model = AutoEncoder()
latent_model.to(device)



Loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(latent_model.parameters(), lr=1e-3)
num_epochs = 300
for epoch in range(num_epochs):
    for batch in dataLoader:
        
        batch = batch.to(device)
        output = latent_model(batch)
        loss = Loss_fn(output, batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()/80:.4f}")


Epoch [1/300], Loss: 45.3436
Epoch [2/300], Loss: 47.5453
Epoch [3/300], Loss: 47.3726
Epoch [4/300], Loss: 50.0244
Epoch [5/300], Loss: 45.1026
Epoch [6/300], Loss: 45.5414
Epoch [7/300], Loss: 48.7860
Epoch [8/300], Loss: 46.8929
Epoch [9/300], Loss: 45.6318
Epoch [10/300], Loss: 45.9316
Epoch [11/300], Loss: 46.8020
Epoch [12/300], Loss: 46.9558
Epoch [13/300], Loss: 45.5334
Epoch [14/300], Loss: 48.1124
Epoch [15/300], Loss: 45.1244
Epoch [16/300], Loss: 45.5952
Epoch [17/300], Loss: 48.3466
Epoch [18/300], Loss: 48.1243
Epoch [19/300], Loss: 48.0347
Epoch [20/300], Loss: 45.6676
Epoch [21/300], Loss: 44.7166
Epoch [22/300], Loss: 46.9204
Epoch [23/300], Loss: 46.0224
Epoch [24/300], Loss: 46.5173
Epoch [25/300], Loss: 47.9117
Epoch [26/300], Loss: 42.8084
Epoch [27/300], Loss: 49.2887
Epoch [28/300], Loss: 45.8082
Epoch [29/300], Loss: 43.8463
Epoch [30/300], Loss: 43.2072
Epoch [31/300], Loss: 46.2820
Epoch [32/300], Loss: 44.7336
Epoch [33/300], Loss: 46.7137
Epoch [34/300], Los

In [30]:
latent_model.eval()
total_mse = 0.0
total_mae = 0.0
num_batches = 0

with torch.no_grad():
    for batch in dataLoader_test:
        batch = batch.to(device)
        output = latent_model(batch)
        total_mse += Loss_fn(output, batch).item()
        total_mae += torch.mean(torch.abs(output - batch)).item()
        num_batches += 1

avg_mse = total_mse / max(1, num_batches)
avg_mae = total_mae / max(1, num_batches)
print(f"Avg MSE: {avg_mse/80:.4f}, Avg MAE: {avg_mae:.4f}")

Avg MSE: 15.3108, Avg MAE: 31.1628
